# Advanced Computational Physics


## More about Python: Functions, Classes and Symbolic computing
### Functional programming: Decorators and a primer on testing


#### *X. Cid Vidal, J.A. Hernando Morata*, in collaboration with *G. Martínez-Lema*, *M. Kekic*.
####  USC, October 2026

In [1]:
import time 
print(' Last revision ', time.asctime())

 Last revision  Sat Oct  3 22:12:59 2026


In [2]:
# general imports
import time
import math
import functools
import numpy as np

This notebook has two parts:

  1. **Decorators**: functions that modify other functions. We already know the ingredients from the notebook about functions: functions passed as arguments, functions returned by functions, and closures.

  2. **Testing**: how to check that a function does what it is expected to do, and the *pytest* module.

## 1. Decorators

### 1.1 What is a decorator?

A decorator is a function that takes a function and returns a new one, usually a closure that adds some behaviour to the original function.

Python has a special syntax for it:

```python
@decorator
def function():
    ...
```

is exactly the same as

```python
def function():
    ...
function = decorator(function)
```

In [3]:
def add_initial_symbol(function):

    def decorate(): return '> ' +  function()
    return decorate

@add_initial_symbol
def hello():

    return 'hello'

hello()

'> hello'

First we define the decorator *add_initial_symbol*, which returns a new function, *decorate*. The line *@add_initial_symbol* on top of the definition of *hello* replaces *hello* by that new function.

The name *hello* now refers to *decorate*, which calls the original *hello* and adds a symbol to its result.

### 1.2 A general decorator: *\*args* and *\*\*kwargs*

*add_initial_symbol* only works for functions without arguments. To decorate any function, the inner function (usually called *wrapper*) accepts *\*args, \*\*kwargs* and passes them on to the original function.

A useful example: a decorator that measures the execution time of a function.

In [4]:
def timer(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        t0 = time.perf_counter()
        result = func(*args, **kwargs)
        print(f'{func.__name__} took {time.perf_counter() - t0:.4f} s')
        return result
    return wrapper

@timer
def harmonic(n):
    """ sum of 1/k for k = 1, ..., n """
    return sum(1 / k for k in range(1, n + 1))

harmonic(1_000_000)

harmonic took 0.0860 s


14.392726722865724

Notice two things:

  * The wrapper must *return* the result of the original function. Otherwise the decorated function always returns *None*.

  * *functools.wraps* (itself a decorator!) copies the name, the docstring, etc. of the original function into the wrapper. Without it, *harmonic.\_\_name\_\_* would be *'wrapper'* and *help(harmonic)* would not show the docstring. Always use it when writing decorators.

### 1.3 Decorators in the standard library

In the notebook about functions we wrote a cache by hand (a closure) to speed up the recursive Fibonacci. *functools.lru_cache* does the same as a decorator:

In [5]:
def fib_rec(n):
    """ n-th Fibonacci number, naive recursion """
    if n < 2:
        return n
    return fib_rec(n-1) + fib_rec(n-2)

@functools.lru_cache(maxsize=None)
def fib_cached(n):
    """ n-th Fibonacci number, recursion with a cache """
    if n < 2:
        return n
    return fib_cached(n-1) + fib_cached(n-2)

for f in (fib_rec, fib_cached):
    t0 = time.perf_counter()
    print(f'{f.__name__}(28) = {f(28)}   took {time.perf_counter() - t0:.6f} s')

print(fib_cached.cache_info())

fib_rec(28) = 317811   took 0.062899 s
fib_cached(28) = 317811   took 0.000023 s
CacheInfo(hits=26, misses=29, maxsize=None, currsize=29)


*functools.cache* is a shorter name for *lru_cache(maxsize=None)*.

Decorators can be stacked. They are applied from bottom to top:

```python
@timer
@functools.cache
def f(x):
    ...
```

is the same as *f = timer(functools.cache(f))*.

Some decorators take arguments, like *@functools.lru_cache(maxsize=None)*. Python first calls *lru_cache(maxsize=None)*, which returns the decorator, and then applies it: *f = lru_cache(maxsize=None)(f)*. *@pytest.mark.parametrize(...)*, at the end of this notebook, works the same way.

We will meet more decorators soon: *@property*, *@classmethod*, *@staticmethod* and *@dataclass* in the notebooks about classes.

## 2. Testing

### 2.1 Testing a function

It is a good practice to write a set of tests to check that a function does what it is expected to do.

A test is just a function, usually named *test_function*, that uses *assert*: *assert condition* does nothing if *condition* is *True*, and raises an *AssertionError* otherwise.

This is the first version of *fib* from the notebook about functions:

In [6]:
def fib(n):
    """ returns a list with the first n numbers of the Fibonacci series
    """
    ns = [0, 1]

    for i in range(2, n):
        ns.append(ns[i-2] + ns[i-1])

    return ns

In [7]:
def test_fib(ns):
             
    assert ns[0] == 0
    assert ns[1] == 1

    n   = len(ns)
    oks = [ns[i] == ns[i-1] + ns[i-2] for i in range(2, n)]
    assert all(oks)

    return True

In [8]:
n      = 20
values = fib(n)
print(values)
print(test_fib(values))
values = [0, 1, 5, 8]
print(values)
try:
    test_fib(values)
except AssertionError:
    print('AssertionError: not a Fibonacci series')

[0, 1, 1, 2, 3, 5, 8, 13, 21, 34, 55, 89, 144, 233, 377, 610, 987, 1597, 2584, 4181]
True
[0, 1, 5, 8]
AssertionError: not a Fibonacci series


*try: ... except AssertionError:* catches the error, so that the notebook does not stop. We will see more about exceptions in the notebooks about classes.

*fib(20)* passes the test, but *fib* still has a bug! *test_fib* never checks the number of elements, and never tries the edge cases $n = 0, 1$. A good test also checks the edge cases:

In [9]:
def test_fib_length():
    for n in range(10):
        assert len(fib(n)) == n, f'fib({n}) has {len(fib(n))} elements'

try:
    test_fib_length()
except AssertionError as error:
    print('test failed:', error)

test failed: fib(0) has 2 elements


The message after the comma in *assert* is shown when the test fails. (In the notebook about functions, *fib* was fixed by returning *ns[:n]*.)

### 2.2 The importance of testing

In fact, a function can be *defined* by its tests. It is a good practice to write the tests of a function before implementing it!

For example, what is the two-argument function that must pass the following tests?

In [10]:
def test_addition(function, a, b, null = 0):

    assert function(a, b) == function(b, a)

    assert function(a, function(a, b)) == function(b, function(a, a))

    assert function(a, null) ==  a

    return True

In [11]:
## j denotes the imaginary unit in python
add   = lambda a, b : a + b
mul   = lambda a, b : a * b
a, b  = 1+0j, 0+1j

for name, function in [('add', add), ('mul', mul)]:
    try:
        test_addition(function, a, b)
        print(name, 'test passed')
    except AssertionError:
        print(name, 'test not passed')

add test passed
mul test not passed


The multiplication is commutative and associative, but its neutral element is 1, not 0. Try *test_addition(mul, a, b, null=1)*.

### 2.3 Testing numerical code

Never compare floating-point numbers with *==*:

In [12]:
print(0.1 + 0.2 == 0.3)
print(0.1 + 0.2)

False
0.30000000000000004


Use a tolerance instead. *math.isclose(a, b, rel_tol=1e-9, abs_tol=0)* checks that $|a - b| \le \max(\mathrm{rel\_tol} \cdot \max(|a|, |b|), \mathrm{abs\_tol})$. For arrays, use *np.isclose* and *np.allclose*.

The default tolerance is relative: to compare with 0 you need *abs_tol*.

In [13]:
print(math.isclose(0.1 + 0.2, 0.3))
print(math.isclose(math.sin(math.pi), 0))                 # relative tolerance and 0: False!
print(math.isclose(math.sin(math.pi), 0, abs_tol=1e-12))

x = np.linspace(0, 1, 5)
print(np.allclose(np.sin(x)**2 + np.cos(x)**2, 1))

True
False
True
True


**Test properties, not just values.** In numerical code we rarely know the exact result for a general input. But we usually know properties that the result must satisfy: exact results for special cases, symmetries, limits, scaling laws, conservation laws... Test those.

Also, choose the tolerance according to the precision of the method: an approximate method will never agree to 15 digits.

Let's test the midpoint integration from the notebook about functions (its error goes like $h^2$):

In [14]:
def integrate(f, a, b, n=100_000):
    """ midpoint rule integral of f between a and b """
    h = (b - a) / n
    return h * sum(f(a + (i + 0.5) * h) for i in range(n))

def test_integrate():
    assert math.isclose(integrate(math.sin, 0, math.pi), 2, rel_tol=1e-8)   # known value
    assert math.isclose(integrate(math.sin, -1, 1), 0, abs_tol=1e-12)       # odd function
    assert math.isclose(integrate(lambda x: x**2, 0, 3), 9, rel_tol=1e-8)   # polynomial

test_integrate()
print('test_integrate passed')

test_integrate passed


### 2.4 Pytest

Writing *try: ... except* around every test and running them one by one is tedious. A testing framework finds the test functions, runs them, catches the failures and reports them.

There are several modules to implement tests in Python. The most common one is [*pytest*](https://docs.pytest.org/) (see the documentation to explore its many capabilities). Its conventions:

  * The tests live in files named *test_\*.py*.
  * The test functions are named *test_\**.
  * They use plain *assert*. When an assertion fails, pytest shows the values involved.

Running *pytest* in a terminal finds and runs all the tests in the directory. (It is not part of the standard library: install it with *conda install pytest*.)

Usually the functions live in a module, and the tests in a separate file. From a notebook, the magic *%%writefile* writes the content of a cell into a file:

In [15]:
%%writefile mymath.py
""" a small module with numerical functions """

def integrate(f, a, b, n=100_000):
    """ midpoint rule integral of f between a and b """
    h = (b - a) / n
    return h * sum(f(a + (i + 0.5) * h) for i in range(n))

def fib(n):
    """ returns a list with the first n numbers of the Fibonacci series """
    ns = [0, 1]
    for i in range(2, n):
        ns.append(ns[i-2] + ns[i-1])
    return ns

Overwriting mymath.py


In [16]:
%%writefile test_mymath.py
import math
import pytest
from mymath import integrate, fib

def test_integrate_sin():
    assert integrate(math.sin, 0, math.pi) == pytest.approx(2)

@pytest.mark.parametrize('n', [0, 1, 2, 10])
def test_fib_length(n):
    assert len(fib(n)) == n

Overwriting test_mymath.py


*pytest.approx(expected)* compares with a relative tolerance of $10^{-6}$ by default.

*@pytest.mark.parametrize* is a decorator with arguments: the test runs once for each value of *n*.

Now we run pytest. The *!* sends the line to the shell, and *sys.executable* makes sure that we use the same Python as the notebook:

In [17]:
import sys
!{sys.executable} -m pytest -q test_mymath.py

.FF..                                                                    [100%]
=================================== FAILURES ===================================
______________________________ test_fib_length[0] ______________________________

n = 0



    @pytest.mark.parametrize('n', [0, 1, 2, 10])
    def test_fib_length(n):
>       assert len(fib(n)) == n
E       assert 2 == 0
E        +  where 2 = len([0, 1])
E        +    where [0, 1] = fib(0)

test_mymath.py:10: AssertionError
______________________________ test_fib_length[1] ______________________________

n = 1

    @pytest.mark.parametrize('n', [0, 1, 2, 10])
    def test_fib_length(n):
>       assert len(fib(n)) == n
E       assert 2 == 1
E        +  where 2 = len([0, 1])
E        +    where [0, 1] = fib(1)

test_mymath.py:10: AssertionError
=========================== short test summary info ============================
FAILED test_mymath.py::test_fib_length[0] - assert 2 == 0
FAILED test_mymath.py::test_fib_length[1] - assert 2 == 1
2 failed, 3 passed in 0.08s


pytest tells us which cases fail, and why: *len(fib(0))* is 2, not 0. Fix *fib* in *mymath.py* (return *ns[:n]*) and run pytest again.

Other things worth knowing (see the documentation):

  * *pytest.raises* checks that a function raises an exception. We will use it in the notebooks about classes.
  * *pytest -k name* runs only the tests whose name contains *name*; *pytest -x* stops at the first failure.
  * The standard library also has testing modules: *doctest* and *unittest*.

## Summary

  * A decorator is a function that takes a function and returns a new one. *@decorator* on top of *def f* is the same as *f = decorator(f)*.

  * A general decorator uses a wrapper with *\*args, \*\*kwargs*, returns the result of the original function, and uses *functools.wraps*.

  * Decorators with arguments need one more level of functions. Stacked decorators are applied from bottom to top.

  * *functools.lru_cache* and *functools.cache* memoize a function.

  * A test uses *assert*. Test the edge cases, and in numerical code test properties, always comparing floats with a tolerance (*math.isclose*, *np.allclose*, *pytest.approx*).

  * pytest finds and runs the *test_\** functions in the *test_\*.py* files. *@pytest.mark.parametrize* runs a test for several inputs.

### Exercises

  1. Write a decorator *count_calls* that counts how many times a function is called, storing the number in an attribute of the wrapper (*wrapper.calls*). Use *functools.wraps*.

      * Apply it to the naive recursive Fibonacci *fib_rec*, and print the number of calls for $n = 10, 20, 25$ (reset *fib_rec.calls = 0* in between).

      * Add *@functools.lru_cache* below *@count_calls*. How many calls are there now? What happens if you swap the order of the two decorators?


  2. Testing the Poisson distribution, $p(n | \mu) = \mu^n e^{-\mu} / n!$.

      * Write a file *poisson.py* with a function *poisson(n, mu)*, and a file *test_poisson.py* with tests that check, for $\mu = 5$:
          * $\sum_{n=0}^{50} p(n|\mu) \simeq 1$,
          * the mean, $\sum_{n=0}^{50} n \, p(n|\mu) \simeq \mu$,
          * $p(0|\mu) = e^{-\mu}$.

        Use *pytest.approx* or *math.isclose*, never *==*.

      * Use *@pytest.mark.parametrize* to run the tests for several values of $\mu$, and run them with pytest.